# Session 25 · Vector databases — a fast index for meaning
**Homework notebook.** In class we drew it, saw it live and ran it on the projector. Here you run every step yourself, then
finish four TODOs in `s25/store.py` and prove them with the checker.

| Part | What | Time |
|---|---|---|
| A | Why a database at all, the words you need, how today fits | 15 min reading |
| B | Run it: load → look at one row → search inside the DB → filter → update one row → brute force at scale → HNSW speed vs accuracy | 35 min |
| C | Your homework file: four TODOs, run, check, commit | 40 min |
| D | Interview questions and what comes next | 10 min |

**Before you start** (course repo folder, PowerShell):
```powershell
git pull
uv sync                                   # NEW today: chromadb (a vector database that runs inside Python)
uv run jupyter lab module04/s25_vector_db.ipynb
```
You need Session 24's model (bge-small) — if you did S24's homework, it's already downloaded and the 53 vectors are cached.

---
# Part A — Why a database at all?

### Where we left off
In Session 24 we kept 53 vectors in a Python variable and compared every question with **all 53**. That's called
**brute force** (or exact / flat search). It's always right, and for 53 vectors it takes microseconds.

Now scale it up:

| System | Vectors | Brute force per question |
|---|---|---|
| Our help centre | 53 | instant |
| A bank's policy library | ~200,000 sections | noticeable |
| An e-commerce catalogue | ~50,000,000 products | far too slow, for every user, every second |

Three more problems with "a Python variable":
1. **It disappears** when the program stops — we'd re-embed everything on every start.
2. **No filtering** — "refunds, but only *payments* articles updated this year" means checking every row by hand.
3. **No updates** — change one article and we rebuild the whole list.

A **vector database** solves all four: it stores vectors **on disk**, keeps **metadata** next to them, supports **add / update / delete**,
and builds an **index** so it can find the nearest vectors without comparing with every one.

### The trade: "approximate"
The index doesn't check every vector, so once in a while it misses one of the true closest matches. That's why it's called
**ANN — approximate nearest neighbour**. You choose how much accuracy to trade for speed, and you **measure** it with **recall@k**.

### The words you need (full list: `module04/terms_s25.md`)

| Term | Plain meaning | Analogy |
|---|---|---|
| **Vector database** | A database built to store vectors and find the nearest ones fast | A library with a catalogue, not a pile of books |
| **Collection** | One named set of rows in the database (like a table) | One shelf: "help centre" |
| **Row / record** | id + vector + metadata + the original text | A library card |
| **Metadata (payload)** | Labels stored next to the vector: category, date, customer, language | The genre and year on the library card |
| **Filter (where)** | Only search rows whose metadata matches | "Only cookery books after 2020" |
| **Brute force / exact search** | Compare with every vector | Reading every book to find one |
| **Index** | A structure that lets you skip most vectors | The catalogue |
| **ANN** | Approximate nearest neighbour: fast, nearly always right | Asking the librarian, not reading every book |
| **HNSW** | The most common ANN index: layers of "highways" and "local roads" between vectors | Expressway → main road → lane |
| **IVF** | Another ANN index: split vectors into neighbourhoods, search only the nearest few | Only searching the nearest districts of a city |
| **ef_search** | How many candidates HNSW looks at per question. Higher = more accurate, slower | How many shops you check before buying |
| **M (max_neighbors)** | How many links each point keeps. Higher = more accurate, more memory | How many roads leave each junction |
| **recall@k** | Share of the TRUE top k that the index found | "Did the librarian miss any?" |
| **Upsert** | Update the row if the id exists, insert it if not | Replace one library card |

### How today is built
```
S24:  53 vectors in a Python list ──► compare with ALL ──► top 3
S25:  53 rows in ChromaDB on disk ──► index (HNSW) + metadata filter ──► top 3
      row = id | vector (384 numbers) | metadata {doc_id, category, updated} | text
```
ChromaDB runs **inside Python** with files on disk — no server, no Docker. Session 26 moves the same data to a **server** database
(Qdrant / pgvector) like companies run in production.

---
# Part B — Run it
### B1. Load the help centre into a vector database

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "module04") if (Path.cwd() / "module04").exists() else str(Path.cwd()))

import numpy as np
import s24_utils as S24
import s25_utils as S25

emb = S24.Embedder("bge-small")
client = S25.open_db()                               # files in module04/.s25_chroma - survive restarts
col = S25.build_collection(client, emb)
print(col.count(), "rows in collection", repr(col.name))

### B2. What one row holds
id + vector + **metadata** + the original text. The metadata comes from `module04/help_centre_meta.json`: every article has a
**category** and the date it was last **updated**. Dates are stored as numbers (20260930) so the database can compare them.

In [ ]:
row = col.get(limit=1, include=["embeddings", "metadatas", "documents"])
print("id      :", row["ids"][0])
print("vector  :", np.round(row["embeddings"][0][:6], 3), "... (", len(row["embeddings"][0]), "numbers )")
print("metadata:", row["metadatas"][0])
print("text    :", row["documents"][0][:100], "...")

**Restart the kernel** (Kernel → Restart), run the first cell of B1 again **without** `build_collection`, and use
`col = client.get_collection("help_centre")` — the rows are still there. That's *persistence*.

### B3. Search inside the database
Same idea as Session 24 — embed the question, find the nearest — but now the **database** does the comparing.
Chroma returns *cosine distance* (= 1 − cosine similarity); `S25.search` turns it back into a similarity score.

In [ ]:
for h in S25.search(col, emb, "how long do refunds take"):
    m = h["meta"]
    print(f"{h['score']:.3f}  {m['title']} > {m['heading']:32} [{m['category']}, updated {m['updated']}]")

### B4. Filters — search only where it makes sense
Real systems almost always filter: only this customer's documents, only this product line, only current policies.
ChromaDB's filter language (`where`):

| You want | `where` |
|---|---|
| one category | `{"category": "payments"}` |
| updated on/after a date | `{"updated": {"$gte": 20260601}}` |
| both | `{"$and": [{"category": "payments"}, {"updated": {"$gte": 20260601}}]}` |

In [ ]:
for where in [None, S25.where_filter("payments"), S25.where_filter("returns", "2026-06-01"), S25.where_filter("products")]:
    hits = S25.search(col, emb, "how long do refunds take", 3, where)
    print(where)
    for h in hits:
        print(f"    {h['score']:.3f}  {h['meta']['doc_id']:15} {h['meta']['category']:9} {h['meta']['updated']}")

The last one asks about refunds **inside the Size guide**. The filter wins: you get size-guide rows with tiny scores.
**Filters are rules, not suggestions** — that's exactly what you want for "only Asha's documents" (security), and why a wrong
filter silently gives bad answers. This is ShopAssist Pro's Gap 1 (whose data?) in RAG form.

### B5. Update one row (upsert) — no full rebuild
The payments team adds a line about no-cost EMI. We re-embed **one** row and upsert it.

In [ ]:
q = "can I pay in instalments with my credit card"
print("before:", [h["meta"]["heading"] for h in S25.search(col, emb, q, 2)])

row = col.get(where={"doc_id": "payments"}, limit=1, include=["metadatas", "documents"])
new_text = row["documents"][0] + " NEW: no-cost EMI in 3, 6 or 9 monthly instalments on all major credit cards."
col.upsert(ids=row["ids"], documents=[new_text], metadatas=row["metadatas"], embeddings=[emb.embed([new_text])[0].tolist()])

print("after :", [h["meta"]["heading"] for h in S25.search(col, emb, q, 2)])
col = S25.build_collection(client, emb)              # put the original text back

### B6. Why an index? Brute force at scale
Fake (but realistically clustered) 384-number vectors — the same size as bge-small's.

In [ ]:
import time
for n in (10_000, 50_000, 200_000):
    x, qs = S25.clustered_vectors(n, n_queries=20)
    t0 = time.perf_counter()
    for qv in qs:
        _ = x @ qv
    print(f"{n:>8,} vectors: {(time.perf_counter() - t0) * 1000 / len(qs):6.2f} ms per question, {x.nbytes / 1e6:5.0f} MB of vectors")

Time and memory grow **in a straight line** with the data. 50 million products × every user × every second doesn't fit.
Rule of thumb: **1 million × 384-dim float32 ≈ 1.5 GB** of raw vectors.

### B7. HNSW: speed vs accuracy, measured
20,000 vectors. First the **true** top 10 for 50 questions (brute force). Then the same questions through HNSW with three settings.
**recall@10** = how many of the true top 10 the index found.

In [ ]:
x, qs = S25.clustered_vectors(20_000)
exact = S25.exact_top_k(x, qs, 10)
ann = S25.open_db(None)                               # in memory, just for the experiment
print(f"{'setting':9} {'ef_search':>9} {'M':>3} {'recall@10':>9} {'ms/query':>9}")
for preset, p in S25.HNSW_PRESETS.items():
    r = S25.hnsw_run(ann, x, qs, preset)
    print(f"{preset:9} {p['ef_search']:>9} {p['max_neighbors']:>3} {S25.recall_at_k(exact, r['ids']):>9.2f} {r['ms_per_query']:>9.2f}")

Read the table like an engineer:
- **fast** misses some true neighbours (recall well below 1) — fine for "similar products you may like", risky for "the policy clause that decides a claim".
- **default** usually finds all of them at this size.
- **accurate** costs more time and memory for little extra here; it pays off at millions of vectors.

**Try it:** run with `S25.clustered_vectors(50_000)` — does `fast` get worse? That's why you re-measure when data grows.

---
# Part C — Your homework: `s25/store.py`

| TODO | What | Lines |
|---|---|---|
| 1 | `make_metadata(chunk, info)` — the labels for one row; date as an integer `YYYYMMDD` | 3–4 |
| 2 | `build_where(category, since)` — turn two optional filters into Chroma's `where` (incl. `$and`) | 6–8 |
| 3 | `recall_at_k(exact, approx, k)` — the metric from B7 | 3–4 |
| 4 | `MY_FILTERED` — two filtered questions of your own (one using both filters) | 2 |

**Steps** (course repo folder, PowerShell):
```powershell
uv run python module04/s25_setup.py                 # copies the file to <your work repo>\s25\store.py
# edit it in VS Code: search for TODO
uv run python "<your work repo>\s25\store.py"       # builds the DB with YOUR metadata, filters with YOUR where,
                                                    # measures HNSW with YOUR recall; writes results.md (~30 s)
uv run pytest tests/test_s25.py -rs                 # the checker (offline)
cd "<your work repo>" ; git add s25 ; git commit -m "S25: vector database" ; git push
```
Each TODO's docstring has a **Check:** line — paste it into this notebook to test before running the whole file.

---
# Part D — Interview questions

| Question | Fresher answer | Experienced answer |
|---|---|---|
| Why use a vector database instead of a list? | It stores vectors on disk, searches fast with an index, and can filter by metadata. | Persistence, ANN indexes (sub-linear search), metadata filtering, CRUD/upsert, replication and sharding, access control — brute force is linear in N per query. |
| What is ANN and why "approximate"? | It finds the nearest vectors fast but can miss a few. | Trades recall for latency by searching a graph/partitions instead of all N; tuned with parameters like ef_search/M (HNSW) or nprobe (IVF), measured with recall@k against exact search. |
| How does HNSW work? | Layers of links between vectors: start on a sparse top layer and zoom down. | A hierarchical small-world graph: greedy search from an entry point on sparse upper layers, refining on dense lower layers; M controls degree/memory, ef_construction build quality, ef_search the query-time candidate list. |
| HNSW vs IVF? | Both skip most vectors; HNSW uses a graph, IVF uses clusters. | HNSW: high recall, low latency, more memory, incremental inserts. IVF: cheaper memory, needs training/re-clustering, recall depends on nprobe; often combined with product quantisation at very large scale. |
| How do you keep one customer from seeing another's documents? | Filter by customer id in the search. | Enforce tenant isolation in the query (metadata filter or per-tenant collection/namespace), never post-filter in app code; consider the filter's effect on ANN recall. |
| How do you update a document? | Re-embed it and upsert. | Upsert by stable id, re-embed only changed chunks, version the embedding model; a model change means a full re-index (blue/green). |

### What's next
**S26 — Production vector stores.** ChromaDB is a library on your laptop. Next: Qdrant, pgvector, Pinecone and Weaviate — how to
choose (the S24 framework again), multi-tenancy, scaling, and migrating this exact collection to a server database.